# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gadarshreddy/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [token]
)

MAR = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("HF_TOKEN found.")
print("Connected to the FlyRank warehouse.")
print("Development month: March 2026")

HF_TOKEN found.
Connected to the FlyRank warehouse.
Development month: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents one pseudonymized content item for one report date.

### Time window

I will use March 2026 (`month=2026-03`) as the development month for this lane. I will treat June 2026 as a sealed final month and will not use it to develop the score or label logic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

I will use observed performance and content signals that are available for the content item at the decision moment, such as impressions, clicks, CTR, average position, sessions, engagement rate, search volume, competition, word count, content age, and days since last update.

### Label / proxy

The lane does not have a directly observed ground-truth label for "CTR / engagement opportunity." I will therefore treat the opportunity score as a decision-support proxy rather than a true outcome label.

### Context

Content type, search demand, competition, content age, and update timing provide context for interpreting performance signals.

### Excluded

I will exclude future-outcome fields and any label-derived fields because they would not be known at the decision moment and could cause data leakage. I will also exclude client-identifying information, raw queries, and URLs.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
result = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
""").df()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [24]:
grain_check = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS rows_per_grain
FROM {MAR}
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,rows_per_grain


In [25]:
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE) AS ga4_available_rows
FROM {MAR}
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,9841378,6822637


In [26]:
features = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {MAR}
WHERE client_has_gsc IS TRUE
LIMIT 1000
""").df()

display(features.head())

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727


### Five features and when they are available

1. **gsc_impressions** — shows how often the content appeared in Google Search; available at the decision moment from the accumulated search-performance data.

2. **gsc_clicks** — shows how many search clicks the content received; available at the decision moment from the accumulated search-performance data.

3. **gsc_avg_position** — shows the content's average Google Search position; available at the decision moment from the search-performance data.

4. **gsc_data_available** — indicates whether usable Google Search Console data is available for the row; available at the decision moment and used to distinguish unavailable search data from a real zero.

5. **report_date** — identifies the date represented by the performance row; available at the decision moment because it defines the observation date.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data is useful for ranking content opportunities, but it cannot tell me why a page performed poorly by itself.

- Client history is uneven, so the same time window does not represent the same amount of historical data for every client.
- Google Analytics data is not available for every row, so missing or unavailable engagement data should not automatically be treated as zero engagement.
- The March 2026 partition is useful for development, but it does not represent future performance or prove that an opportunity score will work on unseen data.
- The data shows measured search and engagement signals, but it does not capture every reason a page may gain or lose performance, such as changes outside the dataset.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.